# 🔍 Extrato Atendimento Central — Gold vs Silver (AWS Athena)

Compares **AWS Athena** production tables:
* Gold: `gold_huntington_prod.clinisys_extrato_atendimentos_central` (plural)
* Silver: `silver_clinisys_prod.view_extrato_atendimentos_central`

All analysis is run directly in AWS Athena to ensure there is no local resource footprint.

### Sections
1. Setup & helpers
2. Schema Audit
3. Row Counts & Date Bounds
4. Key Overlap
5. Yearly Breakdown
6. Field-Level Discrepancies (matched rows)
7. Sample Mismatches per column
8. Sample: Only in Gold
9. Sample: Only in Silver


In [10]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
# Test Athena connectivity and print row counts
total_g = run_query("SELECT count(*) as cnt FROM gold_huntington_prod.clinisys_extrato_atendimentos_central").iloc[0]['cnt']
total_s = run_query("SELECT count(*) as cnt FROM silver_clinisys_prod.view_extrato_atendimentos_central WHERE CAST(data AS DATE) >= DATE '2019-01-01'").iloc[0]['cnt']
print(f'Athena Gold rows:   {total_g:,}')
print(f'Athena Silver rows: {total_s:,} (filtered >= 2019-01-01)')


Athena Gold rows:   784,359
Athena Silver rows: 499,999 (filtered >= 2019-01-01)


## Part 1 — Schema Audit

Column names and types side-by-side in Athena.

In [11]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
gold_schema = run_query("""
    SELECT column_name, data_type AS gold_type
    FROM information_schema.columns
    WHERE table_schema = 'gold_huntington_prod'
      AND table_name   = 'extrato_atendimentos_central'
""")
silver_schema = run_query("""
    SELECT column_name, data_type AS silver_type
    FROM information_schema.columns
    WHERE table_schema = 'silver_clinisys_prod'
      AND table_name   = 'view_extrato_atendimentos_central'
""")

schema_cmp = gold_schema.merge(silver_schema, on='column_name', how='outer')
schema_cmp['match'] = schema_cmp.apply(
    lambda r: '✅' if r['gold_type'] == r['silver_type'] else
              ('➕ gold only' if pd.isna(r['silver_type']) else
               ('➕ silver only' if pd.isna(r['gold_type']) else '⚠️ type diff')), axis=1)
display(schema_cmp)


,column_name,gold_type,silver_type,match
0,_dlt_id,NaN,varchar,➕ silver only
1,agenda,NaN,bigint,➕ silver only
2,agenda_nome,NaN,varchar,➕ silver only
3,agendamento_id,NaN,bigint,➕ silver only
4,bronze_updated_at,NaN,timestamp(6),➕ silver only
5,centro_custos,NaN,bigint,➕ silver only
6,centro_custos_nome,NaN,varchar,➕ silver only
7,chegou,NaN,varchar,➕ silver only
8,confirmado,NaN,bigint,➕ silver only
9,data,NaN,date,➕ silver only


## Part 2 — Row Counts & Date Bounds

In [12]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
gold_stats = run_query("""
    SELECT count(*)                      AS total_rows,
           min(CAST(data AS DATE))       AS min_data,
           max(CAST(data AS DATE))       AS max_data,
           count(DISTINCT prontuario)    AS unique_prontuarios,
           count(DISTINCT agendamento_id) AS unique_agendamentos
    FROM gold_huntington_prod.clinisys_extrato_atendimentos_central
""")
silver_stats = run_query("""
    SELECT count(*)                      AS total_rows,
           min(CAST(data AS DATE))       AS min_data,
           max(CAST(data AS DATE))       AS max_data,
           count(DISTINCT prontuario)    AS unique_prontuarios,
           count(DISTINCT agendamento_id) AS unique_agendamentos
    FROM silver_clinisys_prod.view_extrato_atendimentos_central
    WHERE CAST(data AS DATE) >= DATE '2019-01-01'
""")

stats_cmp = pd.concat([
    gold_stats.assign(source='gold'),
    silver_stats.assign(source='silver (>= 2019-01-01)')
], ignore_index=True)[['source','total_rows','min_data','max_data',
                         'unique_prontuarios','unique_agendamentos']]
display(stats_cmp)


,source,total_rows,min_data,max_data,unique_prontuarios,unique_agendamentos
0,gold,784359,2019-01-01,6203-09-15,66070,784359
1,silver (>= 2019-01-01),499999,2019-01-01,6203-09-15,42547,499999


## Part 3 — Key Overlap

How many `agendamento_id` values are shared, exclusive to gold, or exclusive to silver?

In [13]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
overlap = run_query("""
    WITH g AS (
        SELECT DISTINCT CAST(agendamento_id AS BIGINT) AS id
        FROM gold_huntington_prod.clinisys_extrato_atendimentos_central
    ),
    s AS (
        SELECT DISTINCT CAST(agendamento_id AS BIGINT) AS id
        FROM silver_clinisys_prod.view_extrato_atendimentos_central
        WHERE CAST(data AS DATE) >= DATE '2019-01-01'
    )
    SELECT 'In both (overlap)'  AS category, count(*) AS count
    FROM g JOIN s ON g.id = s.id
    UNION ALL
    SELECT 'Only in Gold',  count(*) FROM g WHERE id NOT IN (SELECT id FROM s)
    UNION ALL
    SELECT 'Only in Silver', count(*) FROM s WHERE id NOT IN (SELECT id FROM g)
""")
display(overlap)


,category,count
0,Only in Silver,0
1,In both (overlap),499999
2,Only in Gold,284360


## Part 4 — Yearly Breakdown

In [14]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
yearly = run_query("""
    WITH g AS (
        SELECT CAST(agendamento_id AS BIGINT) AS id,
               YEAR(CAST(data AS DATE))       AS yr
        FROM gold_huntington_prod.clinisys_extrato_atendimentos_central
    ),
    s AS (
        SELECT CAST(agendamento_id AS BIGINT) AS id,
               YEAR(CAST(data AS DATE))       AS yr
        FROM silver_clinisys_prod.view_extrato_atendimentos_central
        WHERE CAST(data AS DATE) >= DATE '2019-01-01'
    )
    SELECT
        COALESCE(g.yr, s.yr)                                        AS year,
        COUNT(DISTINCT g.id)                                         AS gold_total,
        COUNT(DISTINCT s.id)                                         AS silver_total,
        COUNT(DISTINCT CASE WHEN s.id IS NULL THEN g.id END)        AS only_in_gold,
        COUNT(DISTINCT CASE WHEN g.id IS NULL THEN s.id END)        AS only_in_silver
    FROM g FULL OUTER JOIN s ON g.id = s.id
    GROUP BY 1 ORDER BY 1
""")
print('NOTE: only_in_gold = in gold but NOT in silver; only_in_silver = in silver but NOT in gold')
display(yearly)


NOTE: only_in_gold = in gold but NOT in silver; only_in_silver = in silver but NOT in gold


,year,gold_total,silver_total,only_in_gold,only_in_silver
0,2019,49015,61,48954,0
1,2020,34267,427,33840,0
2,2021,92816,2757,90059,0
3,2022,121537,10932,110605,0
4,2023,128634,127734,900,0
5,2024,128382,128381,1,0
6,2025,134900,134900,0,0
7,2026,93400,93399,1,0
8,2027,1398,1398,0,0
9,2028,3,3,0,0


## Part 5 — Field-Level Discrepancies (Matched Rows)

Counts rows where gold and silver differ for each column, on the inner-joined set in Athena.
We normalise values on both sides (using Presto string operations like `lower(trim(col))` and dates/times format formatting).

In [15]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
NORM_G = """
    SELECT
        CAST(agendamento_id AS BIGINT)                AS id,
        CAST(data AS DATE)                             AS data,
        substr(cast(inicio as varchar), 12, 5)         AS inicio,
        CAST(data_agendamento_original AS DATE)        AS data_ag_orig,
        CAST(medico AS BIGINT)                        AS medico,
        CAST(medico2 AS BIGINT)                       AS medico2,
        CAST(prontuario AS BIGINT)                    AS prontuario,
        CAST(evento AS BIGINT)                        AS evento,
        lower(trim(evento2))                           AS evento2,
        CAST(centro_custos AS BIGINT)                 AS centro_custos,
        CAST(agenda AS BIGINT)                        AS agenda,
        lower(trim(chegou))                            AS chegou,
        CAST(confirmado AS BIGINT)                    AS confirmado,
        CAST(paciente_codigo AS BIGINT)               AS paciente_codigo,
        lower(trim(paciente_nome))                     AS paciente_nome,
        lower(trim(medico_nome))                       AS medico_nome,
        lower(trim(medico_sobrenome))                  AS medico_sobrenome,
        lower(trim(medico2_nome))                      AS medico2_nome,
        lower(trim(centro_custos_nome))                AS centro_custos_nome,
        lower(trim(agenda_nome))                       AS agenda_nome,
        lower(trim(procedimento_nome))                 AS procedimento_nome
    FROM gold_huntington_prod.clinisys_extrato_atendimentos_central
"""

NORM_S = """
    SELECT
        CAST(agendamento_id AS BIGINT)                AS id,
        CAST(data AS DATE)                             AS data,
        substr(cast(inicio as varchar), 12, 5)         AS inicio,
        CAST(data_agendamento_original AS DATE)        AS data_ag_orig,
        CAST(medico AS BIGINT)                        AS medico,
        CAST(medico2 AS BIGINT)                       AS medico2,
        CAST(prontuario AS BIGINT)                    AS prontuario,
        CAST(evento AS BIGINT)                        AS evento,
        lower(trim(evento2))                           AS evento2,
        CAST(centro_custos AS BIGINT)                 AS centro_custos,
        CAST(agenda AS BIGINT)                        AS agenda,
        lower(trim(chegou))                            AS chegou,
        CAST(confirmado AS BIGINT)                    AS confirmado,
        CAST(paciente_codigo AS BIGINT)               AS paciente_codigo,
        lower(trim(paciente_nome))                     AS paciente_nome,
        lower(trim(medico_nome))                       AS medico_nome,
        lower(trim(medico_sobrenome))                  AS medico_sobrenome,
        lower(trim(medico2_nome))                      AS medico2_nome,
        lower(trim(centro_custos_nome))                AS centro_custos_nome,
        lower(trim(agenda_nome))                       AS agenda_nome,
        lower(trim(procedimento_nome))                 AS procedimento_nome
    FROM silver_clinisys_prod.view_extrato_atendimentos_central
    WHERE CAST(data AS DATE) >= DATE '2019-01-01'
"""

DISC_COLS = [
    'data','inicio','data_ag_orig','medico','medico2','prontuario','evento','evento2',
    'centro_custos','agenda','chegou','confirmado','paciente_codigo',
    'paciente_nome','medico_nome','medico_sobrenome','medico2_nome',
    'centro_custos_nome','agenda_nome','procedimento_nome'
]

disc_sql = f"""
    WITH g AS ({NORM_G}),
         s AS ({NORM_S}),
    matched AS (SELECT g.id, {', '.join(f'(g.{c} IS DISTINCT FROM s.{c}) AS d_{c}' for c in DISC_COLS)}
                FROM g JOIN s ON g.id = s.id)
    SELECT count(*) AS matched_rows,
           {', '.join(f'SUM(CASE WHEN d_{c} THEN 1 ELSE 0 END) AS {c}' for c in DISC_COLS)}
    FROM matched
"""

disc_wide = run_query(disc_sql)
matched_rows = int(disc_wide['matched_rows'].iloc[0])
disc = disc_wide.drop(columns='matched_rows').T.reset_index()
disc.columns = ['column', 'n_different']
disc['pct_different'] = (disc['n_different'] / matched_rows * 100).round(2)
disc = disc.sort_values('n_different', ascending=False).reset_index(drop=True)

print(f'Matched rows (inner join on agendamento_id): {matched_rows:,}')
print('\n--- Field Discrepancy Summary ---')
display(disc[disc['n_different'] > 0])


Matched rows (inner join on agendamento_id): 499,999

--- Field Discrepancy Summary ---


,column,n_different,pct_different
0,medico_sobrenome,62,0.01
1,medico_nome,62,0.01


## Part 6 — Sample Mismatches per Discrepant Column

In [16]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
COL_EXPRS = {
    'data':                     ('CAST(g.data AS DATE)',             'CAST(s.data AS DATE)'),
    'inicio':                   ("substr(cast(g.inicio as varchar), 12, 5)",
                                  "substr(cast(s.inicio as varchar), 12, 5)"),
    'data_ag_orig':             ('CAST(g.data_agendamento_original AS DATE)',
                                  'CAST(s.data_agendamento_original AS DATE)'),
    'medico':                   ('CAST(g.medico AS BIGINT)',         'CAST(s.medico AS BIGINT)'),
    'medico2':                  ('CAST(g.medico2 AS BIGINT)',        'CAST(s.medico2 AS BIGINT)'),
    'prontuario':               ('CAST(g.prontuario AS BIGINT)',     'CAST(s.prontuario AS BIGINT)'),
    'evento':                   ('CAST(g.evento AS BIGINT)',         'CAST(s.evento AS BIGINT)'),
    'evento2':                  ('g.evento2',                        's.evento2'),
    'centro_custos':            ('CAST(g.centro_custos AS BIGINT)',  'CAST(s.centro_custos AS BIGINT)'),
    'agenda':                   ('CAST(g.agenda AS BIGINT)',         'CAST(s.agenda AS BIGINT)'),
    'chegou':                   ('g.chegou',                         's.chegou'),
    'confirmado':               ('CAST(g.confirmado AS BIGINT)',     'CAST(s.confirmado AS BIGINT)'),
    'paciente_codigo':          ('CAST(g.paciente_codigo AS BIGINT)','CAST(s.paciente_codigo AS BIGINT)'),
    'paciente_nome':            ('g.paciente_nome',                  's.paciente_nome'),
    'medico_nome':              ('g.medico_nome',                    's.medico_nome'),
    'medico_sobrenome':         ('g.medico_sobrenome',               's.medico_sobrenome'),
    'medico2_nome':             ('g.medico2_nome',                   's.medico2_nome'),
    'centro_custos_nome':       ('g.centro_custos_nome',             's.centro_custos_nome'),
    'agenda_nome':              ('g.agenda_nome',                    's.agenda_nome'),
    'procedimento_nome':        ('g.procedimento_nome',              's.procedimento_nome'),
}

top_cols = disc[disc['n_different'] > 0].head(8)['column'].tolist()

for col in top_cols:
    if col not in COL_EXPRS:
        continue
    ge, se = COL_EXPRS[col]
    q = f"""
        SELECT CAST(g.agendamento_id AS BIGINT) AS agendamento_id,
               {ge} AS {col}_gold,
               {se} AS {col}_silver
        FROM gold_huntington_prod.clinisys_extrato_atendimentos_central g
        JOIN silver_clinisys_prod.view_extrato_atendimentos_central s
          ON CAST(g.agendamento_id AS BIGINT) = CAST(s.agendamento_id AS BIGINT)
         AND CAST(s.data AS DATE) >= DATE '2019-01-01'
        WHERE {ge} IS DISTINCT FROM {se}
        LIMIT 5
    """
    samples = run_query(q)
    n = int(disc.loc[disc.column == col, 'n_different'].iloc[0])
    print(f'\n--- [{col}] {n:,} mismatched rows ---')
    display(samples)



--- [medico_sobrenome] 62 mismatched rows ---


,agendamento_id,medico_sobrenome_gold,medico_sobrenome_silver
0,1540124,None,Moreira
1,1540079,None,Moreira
2,1539881,None,Moreira
3,1539769,None,Moreira
4,1539754,None,Bancillon



--- [medico_nome] 62 mismatched rows ---


,agendamento_id,medico_nome_gold,medico_nome_silver
0,1540124,None,Wanderson
1,1540079,None,Wanderson
2,1539881,None,Wanderson
3,1539769,None,Wanderson
4,1539754,None,Ana Leonor


## Part 7 — Sample Rows: Only in Gold

Appointments in `gold` with **no corresponding row** in silver.

In [17]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
gold_only = run_query("""
    SELECT CAST(g.agendamento_id AS BIGINT) AS agendamento_id,
           CAST(g.data AS DATE)              AS data,
           CAST(g.paciente_codigo AS BIGINT) AS paciente_codigo,
           CAST(g.medico AS BIGINT)          AS medico,
           CAST(g.evento AS BIGINT)          AS evento,
           g.centro_custos_nome,
           g.agenda_nome,
           g.procedimento_nome
    FROM gold_huntington_prod.clinisys_extrato_atendimentos_central g
    WHERE NOT EXISTS (
        SELECT 1
        FROM silver_clinisys_prod.view_extrato_atendimentos_central s
        WHERE CAST(s.agendamento_id AS BIGINT) = CAST(g.agendamento_id AS BIGINT)
          AND CAST(s.data AS DATE) >= DATE '2019-01-01'
    )
    ORDER BY CAST(g.data AS DATE) DESC
    LIMIT 20
""")
print('Sample rows only in Gold (newest first; see Part 4 yearly table for totals):')
display(gold_only)


Sample rows only in Gold (newest first; see Part 4 yearly table for totals):


,agendamento_id,data,paciente_codigo,medico,evento,centro_custos_nome,agenda_nome,procedimento_nome
0,512998,2026-04-18,NaN,1183.0,765180,6. HTT Belo Horizonte,Leci Amorim - BELO HORIZONTE,BLOQUEADO ***
1,784772,2024-09-30,NaN,NaN,71,None,None,None
2,817521,2023-11-06,790105.0,3554.0,76089,1. HTT SP - Ibirapuera,Eduardo Leme Alves da Motta - IBIRAPUERA,1ª Consulta Reprodução Humana - IB ***
3,784065,2023-09-26,NaN,NaN,71,None,None,None
4,650951,2023-09-14,NaN,NaN,71,None,None,None
5,793690,2023-08-25,177978.0,3765.0,79082,1. HTT SP - Ibirapuera,Thais Sanches Domingues - IBIRAPUERA,Consulta por Telemedicina de Reprodução Humana...
6,793696,2023-08-25,177978.0,3765.0,79082,1. HTT SP - Ibirapuera,Guilherme Wood - IBIRAPUERA,Consulta por Telemedicina de Reprodução Humana...
7,814873,2023-08-23,147165.0,NaN,216,None,None,None
8,809798,2023-08-15,511773.0,NaN,216,None,None,None
9,809962,2023-08-15,159912.0,NaN,216,None,None,None


## Part 8 — Sample Rows: Only in Silver

Appointments in `silver` **not yet loaded into gold**.

In [ ]:
import pandas as pd, re, warnings
from pyathena import connect
warnings.filterwarnings('ignore')

def run_query(q):
    conn = connect(region_name='sa-east-1', work_group='datalake-admins')
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()
silver_only = run_query("""
    SELECT CAST(s.agendamento_id AS BIGINT) AS agendamento_id,
           CAST(s.data AS DATE)              AS data,
           CAST(s.paciente_codigo AS BIGINT) AS paciente_codigo,
           CAST(s.medico AS BIGINT)          AS medico,
           CAST(s.evento AS BIGINT)          AS evento,
           s.centro_custos_nome,
           s.agenda_nome,
           s.procedimento_nome
    FROM silver_clinisys_prod.view_extrato_atendimentos_central s
    WHERE CAST(s.data AS DATE) >= DATE '2019-01-01'
      AND NOT EXISTS (
        SELECT 1
        FROM gold_huntington_prod.clinisys_extrato_atendimentos_central g
        WHERE CAST(g.agendamento_id AS BIGINT) = CAST(s.agendamento_id AS BIGINT)
    )
    ORDER BY CAST(s.data AS DATE) DESC
    LIMIT 20
""")
print('Sample rows only in Silver (most recent first; see Part 4 yearly table for totals):')
display(silver_only)


Sample rows only in Silver (most recent first; see Part 4 yearly table for totals):


,agendamento_id,data,paciente_codigo,medico,evento,centro_custos_nome,agenda_nome,procedimento_nome


: 